# Problema 4: Clasificación de Smart Cities
- Isaac David Jácome García

## Paso 1: Comprensión del Problema

### Contexto del Problema
Una organización internacional desea clasificar ciudades según su nivel de digitalización, infraestructura tecnológica y servicios inteligentes.

### Objetivo del Análisis
Identificar perfiles de ciudades inteligentes según conectividad, infraestructura IoT, servicios digitales y seguridad.

### Tipo de Aprendizaje
- **Aprendizaje No Supervisado**: No hay etiquetas predefinidas. El SOM descubrirá patrones y agrupamientos automáticamente.

### Variables Relevantes
- **Conectividad**: cobertura_internet, puntos_wifi, alfabetizacion_digital
- **Infraestructura IoT**: sensores_iot, camaras_seguridad, semaforos_inteligentes
- **Servicios digitales**: tramites_virtuales, apps_ciudadanas, consumo_energia_inteligente
- **Movilidad**: indice_movilidad
- **Seguridad**: seguridad_digital
- **Inversión**: inversion_tic

### ¿Por qué SOM?
- Permite descubrir patrones de digitalización sin etiquetas previas
- Conserva relaciones topológicas (ciudades similares estarán cerca en el mapa)
- Reducción de dimensionalidad: 12 variables → mapa 2D visualizable
- Identifica clusters naturales de ciudades con perfiles similares
- Útil para segmentación y descubrimiento de perfiles de smart cities

## Paso 2: Carga del Dataset

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
np.set_printoptions(precision=4, suppress=True)

In [ ]:
with open('../jsons/dataset_som_4.json', 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("=== INFORMACIÓN DEL DATASET ===")
print(f"\nDimensiones: {df.shape}")
print(f"  - Número de patrones (ciudades): {df.shape[0]}")
print(f"  - Número de variables: {df.shape[1]}")

print("\n=== TIPOS DE DATOS ===")
print(df.dtypes)

print("\n=== PRIMERAS 10 FILAS ===")
print(df.head(10))

In [ ]:
print("=== ESTADÍSTICAS DESCRIPTIVAS ===")
print(df.describe())

## Paso 3: Análisis Exploratorio de Datos (EDA)

In [ ]:
print("=== VERIFICACIÓN DE VALORES NULOS ===")
print(df.isnull().sum())
print(f"\nTotal de valores nulos: {df.isnull().sum().sum()}")

In [ ]:
print("=== MATRIZ DE CORRELACIÓN ===")
correlation_matrix = df.corr()
print(correlation_matrix)

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlación - Variables del Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("=== HISTOGRAMAS DE VARIABLES ===")
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, column in enumerate(df.columns):
    axes[i].hist(df[column], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(column, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frecuencia')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("=== BOXPLOTS PARA DETECCIÓN DE OUTLIERS ===")
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, column in enumerate(df.columns):
    axes[i].boxplot(df[column], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='blue'),
                    medianprops=dict(color='red', linewidth=2),
                    whiskerprops=dict(color='blue', linewidth=1.5),
                    capprops=dict(color='blue', linewidth=1.5))
    axes[i].set_title(column, fontsize=10, fontweight='bold')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("=== ANÁLISIS DE DISPERSIÓN: VARIABLES CLAVE vs INVERSIÓN ===")
key_vars = ['cobertura_internet', 'sensores_iot', 'tramites_virtuales', 
            'apps_ciudadanas', 'semaforos_inteligentes', 'indice_movilidad']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, var in enumerate(key_vars):
    axes[i].scatter(df[var], df['inversion_tic'], alpha=0.5, color='steelblue', s=50)
    axes[i].set_xlabel(var, fontsize=10)
    axes[i].set_ylabel('inversion_tic', fontsize=10)
    axes[i].set_title(f'{var} vs inversion_tic', fontsize=11, fontweight='bold')
    axes[i].grid(True, alpha=0.3)
    
    corr = df[var].corr(df['inversion_tic'])
    axes[i].text(0.05, 0.95, f'Corr: {corr:.3f}', transform=axes[i].transAxes,
                fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Paso 4: Preprocesamiento

In [ ]:
print("=== PREPROCESAMIENTO ===")
print("Variables a utilizar (todas son relevantes):")
print(list(df.columns))
print(f"\nNúmero de variables: {len(df.columns)}")

In [ ]:
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(df)
X = X_normalized

print("=== NORMALIZACIÓN MINMAXSCALER ===")
print(f"Rango antes de normalizar:")
print(f"  Mínimo: {df.values.min():.4f}")
print(f"  Máximo: {df.values.max():.4f}")
print(f"\nRango después de normalizar:")
print(f"  Mínimo: {X.min():.4f}")
print(f"  Máximo: {X.max():.4f}")
print(f"\nDimensiones de la matriz X: {X.shape}")

In [ ]:
print("=== VALIDACIÓN DE RANGOS ===")
print(f"Todos los valores están en [0, 1]: {(X >= 0).all() and (X <= 1).all()}")
print(f"\nEstadísticas de la matriz normalizada:")
print(f"  Media: {X.mean():.4f}")
print(f"  Desviación estándar: {X.std():.4f}")
print(f"  Mínimo: {X.min():.4f}")
print(f"  Máximo: {X.max():.4f}")

## Paso 5: Construcción de la Red SOM

In [ ]:
n_entradas = X.shape[1]
map_size_x = 8
map_size_y = 8
n_neuronas = map_size_x * map_size_y

learning_rate = 0.5
coeficiente_vecindad = 1.0
num_iteraciones = 1000

print("=== CONFIGURACIÓN DE LA RED SOM ===")
print(f"Número de entradas: {n_entradas}")
print(f"Tamaño del mapa: {map_size_x}x{map_size_y}")
print(f"Número de neuronas: {n_neuronas}")
print(f"\nParámetros:")
print(f"  Learning rate: {learning_rate}")
print(f"  Coeficiente de vecindad: {coeficiente_vecindad}")
print(f"  Iteraciones: {num_iteraciones}")

n_patrones = X.shape[0]
map_size_recomendado = int(5 * np.sqrt(n_patrones))
print(f"\nJustificación del tamaño del mapa:")
print(f"  Regla 5*sqrt(n): 5*sqrt({n_patrones}) = {map_size_recomendado}")
print(f"  Tamaño seleccionado: {n_neuronas} (8x8)")
print(f"  El tamaño seleccionado es apropiado para {n_patrones} patrones")

In [ ]:
def indice_a_coordenadas(indice, map_size_x):
    x = indice % map_size_x
    y = indice // map_size_x
    return x, y

def distancia_mapa(coord1, coord2):
    return np.sqrt((coord1[0] - coord2[0])**2 + (coord1[1] - coord2[1])**2)

def encontrar_vecinas(indice_ganadora, coeficiente_vecindad, map_size_x, map_size_y):
    coord_ganadora = indice_a_coordenadas(indice_ganadora, map_size_x)
    vecinas = []
    
    for i in range(map_size_x * map_size_y):
        coord = indice_a_coordenadas(i, map_size_x)
        dist = distancia_mapa(coord_ganadora, coord)
        if dist <= coeficiente_vecindad and i != indice_ganadora:
            vecinas.append(i)
    
    return vecinas

print("=== FUNCIONES AUXILIARES DEFINIDAS ===")

In [ ]:
np.random.seed(42)
W = np.random.uniform(0, 1, (n_entradas, n_neuronas))

print("=== INICIALIZACIÓN DE PESOS ===")
print(f"Dimensiones de la matriz de pesos: {W.shape}")
print(f"  - {n_entradas} entradas (filas)")
print(f"  - {n_neuronas} neuronas (columnas)")
print(f"\nRango de pesos iniciales:")
print(f"  Mínimo: {W.min():.4f}")
print(f"  Máximo: {W.max():.4f}")

In [ ]:
print("=== ENTRENAMIENTO DEL SOM ===")
print(f"Iteraciones: {num_iteraciones}")
print(f"Patrones: {X.shape[0]}")
print(f"Learning rate inicial: {learning_rate}")
print(f"Coeficiente de vecindad: {coeficiente_vecindad}\n")

dm_historia = []
W_historia = []
winner_indices = np.zeros(X.shape[0], dtype=int)

W_historia.append(W.copy())

for iteracion in range(1, num_iteraciones + 1):
    lr_actual = learning_rate * (1 - iteracion / num_iteraciones)
    cv_actual = coeficiente_vecindad * (1 - iteracion / num_iteraciones)
    
    distancias_vencedoras = []
    
    for p in range(X.shape[0]):
        patron = X[p]
        
        distancias = np.zeros(n_neuronas)
        for n in range(n_neuronas):
            distancias[n] = np.sqrt(np.sum((patron - W[:, n])**2))
        
        indice_ganador = np.argmin(distancias)
        dist_ganadora = distancias[indice_ganador]
        distancias_vencedoras.append(dist_ganadora)
        winner_indices[p] = indice_ganador
        
        indices_vecinas = encontrar_vecinas(indice_ganador, cv_actual, map_size_x, map_size_y)
        
        W[:, indice_ganador] += lr_actual * (patron - W[:, indice_ganador])
        
        for idx in indices_vecinas:
            W[:, idx] += lr_actual * (patron - W[:, idx])
    
    dm = np.mean(distancias_vencedoras)
    dm_historia.append(dm)
    
    if iteracion % 100 == 0:
        W_historia.append(W.copy())
    
    if iteracion % 100 == 0:
        print(f"Iteración {iteracion}/{num_iteraciones} - Dm: {dm:.6f} - LR: {lr_actual:.4f} - CV: {cv_actual:.4f}")

W_historia.append(W.copy())

print(f"\n=== ENTRENAMIENTO COMPLETADO ===")
print(f"Dm final: {dm_historia[-1]:.6f}")
print(f"Dm inicial: {dm_historia[0]:.6f}")
print(f"Reducción: {((dm_historia[0] - dm_historia[-1]) / dm_historia[0] * 100):.2f}%")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(dm_historia, linewidth=2, color='steelblue')
plt.xlabel('Iteración', fontsize=12)
plt.ylabel('Dm (Distancia promedio)', fontsize=12)
plt.title('Evolución del Dm durante el entrenamiento', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Paso 6: Visualización Topológica

In [ ]:
def calcular_umatrix(W, map_size_x, map_size_y):
    umatrix = np.zeros((map_size_x, map_size_y))
    
    for i in range(map_size_x):
        for j in range(map_size_y):
            idx = i + j * map_size_x
            pesos_centro = W[:, idx]
            vecinos = []
            
            if j > 0:
                idx_arriba = i + (j - 1) * map_size_x
                vecinos.append(W[:, idx_arriba])
            
            if j < map_size_y - 1:
                idx_abajo = i + (j + 1) * map_size_x
                vecinos.append(W[:, idx_abajo])
            
            if i > 0:
                idx_izq = (i - 1) + j * map_size_x
                vecinos.append(W[:, idx_izq])
            
            if i < map_size_x - 1:
                idx_der = (i + 1) + j * map_size_x
                vecinos.append(W[:, idx_der])
            
            if len(vecinos) > 0:
                distancias = [np.sqrt(np.sum((pesos_centro - v)**2)) for v in vecinos]
                umatrix[i, j] = np.mean(distancias)
    
    return umatrix

umatrix = calcular_umatrix(W, map_size_x, map_size_y)

plt.figure(figsize=(10, 8))
sns.heatmap(umatrix.T, cmap='YlOrRd', annot=False, cbar=True,
            square=True, linewidths=0.5)
plt.title('U-Matrix (Matriz de Distancias Unificadas)', fontsize=14, fontweight='bold')
plt.xlabel('X', fontsize=12)
plt.ylabel('Y', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print("=== MAPAS DE CALOR POR VARIABLE (COMPONENT PLANES) ===")

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for i, column in enumerate(df.columns):
    pesos_var = W[i, :].reshape(map_size_x, map_size_y)
    
    sns.heatmap(pesos_var.T, cmap='coolwarm', annot=False, cbar=True,
                square=True, linewidths=0.5, ax=axes[i])
    axes[i].set_title(column, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('X')
    axes[i].set_ylabel('Y')

plt.tight_layout()
plt.show()

In [ ]:
hit_matrix = np.zeros((map_size_x, map_size_y))

for idx in winner_indices:
    x, y = indice_a_coordenadas(idx, map_size_x)
    hit_matrix[x, y] += 1

plt.figure(figsize=(10, 8))
sns.heatmap(hit_matrix.T, cmap='Blues', annot=True, fmt='g', cbar=True,
            square=True, linewidths=0.5)
plt.title('Distribución de Neuronas Ganadoras (Hit Histogram)', fontsize=14, fontweight='bold')
plt.xlabel('X', fontsize=12)
plt.ylabel('Y', fontsize=12)
plt.tight_layout()
plt.show()

print(f"\n=== ESTADÍSTICAS DE HIT HISTOGRAM ===")
print(f"Total de patrones: {np.sum(hit_matrix)}")
print(f"Neuronas con hits: {np.sum(hit_matrix > 0)}")
print(f"Neuronas sin hits: {np.sum(hit_matrix == 0)}")
print(f"Máximo de hits en una neurona: {int(hit_matrix.max())}")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.heatmap(umatrix.T, cmap='YlOrRd', annot=False, cbar=True,
            square=True, linewidths=0.5)
plt.title('U-Matrix', fontsize=12, fontweight='bold')
plt.xlabel('X')
plt.ylabel('Y')

plt.subplot(1, 2, 2)
sns.heatmap(hit_matrix.T, cmap='Blues', annot=False, cbar=True,
            square=True, linewidths=0.5)
plt.title('Hit Histogram', fontsize=12, fontweight='bold')
plt.xlabel('X')
plt.ylabel('Y')

plt.tight_layout()
plt.show()

## Paso 7: Interpretación de Resultados

In [ ]:
print("=== IDENTIFICACIÓN DE CLUSTERS ===")

umatrix_flat = umatrix.flatten()
threshold = np.percentile(umatrix_flat, 50)

cluster_neurons = np.where(umatrix_flat < threshold)[0]
print(f"Neuronas en regiones de baja distancia (potenciales clusters): {len(cluster_neurons)}")
print(f"Umbral utilizado: {threshold:.4f}")

unique_winners, counts = np.unique(winner_indices, return_counts=True)
print(f"\n=== DISTRIBUCIÓN DE PATRONES POR NEURONA ===")
print(f"Neuronas activas: {len(unique_winners)} de {n_neuronas}")
print(f"\nTop 10 neuronas con más patrones:")
top_indices = np.argsort(counts)[-10:][::-1]
for idx in top_indices:
    neuron = unique_winners[idx]
    x, y = indice_a_coordenadas(neuron, map_size_x)
    print(f"  Neurona ({x},{y}) [índice {neuron}]: {counts[idx]} patrones")

In [ ]:
print("=== ANÁLISIS DE PERFILES POR NEURONA GANADORA ===")

top_5_neurons = unique_winners[top_indices[:5]]

for neuron in top_5_neurons:
    x, y = indice_a_coordenadas(neuron, map_size_x)
    count = counts[np.where(unique_winners == neuron)[0][0]]
    
    print(f"\n--- Neurona ({x},{y}) [índice {neuron}] - {count} patrones ---")
    
    pattern_indices = np.where(winner_indices == neuron)[0]
    cluster_data = df.iloc[pattern_indices]
    
    print("Características promedio del cluster:")
    for col in df.columns:
        mean_val = cluster_data[col].mean()
        global_mean = df[col].mean()
        diff = mean_val - global_mean
        symbol = "↑" if diff > 0 else "↓"
        print(f"  {col}: {mean_val:.2f} ({symbol} {abs(diff):.2f} vs global {global_mean:.2f})")

In [ ]:
print("=== IDENTIFICACIÓN DE PERFILES DE CIUDADES ===")

def encontrar_perfil(objetivo, W, df_columns, map_size_x):
    objetivo_norm = scaler.transform([objetivo])[0]
    distancias = np.zeros(n_neuronas)
    for n in range(n_neuronas):
        distancias[n] = np.sqrt(np.sum((objetivo_norm - W[:, n])**2))
    neurona_cercana = np.argmin(distancias)
    return neurona_cercana

# Perfil: Smart City avanzada
perfil_avanzada = [
    df['cobertura_internet'].max(),
    df['camaras_seguridad'].max(),
    df['sensores_iot'].max(),
    df['tramites_virtuales'].max(),
    df['consumo_energia_inteligente'].max(),
    df['inversion_tic'].max(),
    df['alfabetizacion_digital'].max(),
    df['puntos_wifi'].max(),
    df['apps_ciudadanas'].max(),
    df['semaforos_inteligentes'].max(),
    df['indice_movilidad'].max(),
    df['seguridad_digital'].max()
]

neurona_avanzada = encontrar_perfil(perfil_avanzada, W, df.columns, map_size_x)
x, y = indice_a_coordenadas(neurona_avanzada, map_size_x)
print(f"Perfil 'Smart City Avanzada' → Neurona ({x},{y})")

# Perfil: Ciudad tradicional
perfil_tradicional = [
    df['cobertura_internet'].min(),
    df['camaras_seguridad'].min(),
    df['sensores_iot'].min(),
    df['tramites_virtuales'].min(),
    df['consumo_energia_inteligente'].min(),
    df['inversion_tic'].min(),
    df['alfabetizacion_digital'].min(),
    df['puntos_wifi'].min(),
    df['apps_ciudadanas'].min(),
    df['semaforos_inteligentes'].min(),
    df['indice_movilidad'].min(),
    df['seguridad_digital'].min()
]

neurona_tradicional = encontrar_perfil(perfil_tradicional, W, df.columns, map_size_x)
x, y = indice_a_coordenadas(neurona_tradicional, map_size_x)
print(f"Perfil 'Ciudad Tradicional' → Neurona ({x},{y})")

# Perfil: Ciudad en desarrollo
perfil_desarrollo = [
    df['cobertura_internet'].mean(),
    df['camaras_seguridad'].mean(),
    df['sensores_iot'].mean(),
    df['tramites_virtuales'].mean(),
    df['consumo_energia_inteligente'].mean(),
    df['inversion_tic'].mean(),
    df['alfabetizacion_digital'].mean(),
    df['puntos_wifi'].mean(),
    df['apps_ciudadanas'].mean(),
    df['semaforos_inteligentes'].mean(),
    df['indice_movilidad'].mean(),
    df['seguridad_digital'].mean()
]

neurona_desarrollo = encontrar_perfil(perfil_desarrollo, W, df.columns, map_size_x)
x, y = indice_a_coordenadas(neurona_desarrollo, map_size_x)
print(f"Perfil 'Ciudad en Desarrollo' → Neurona ({x},{y})")

# Perfil: Ciudad con alta movilidad
perfil_movilidad = [
    df['cobertura_internet'].max(),
    df['camaras_seguridad'].max(),
    df['sensores_iot'].max(),
    df['tramites_virtuales'].max(),
    df['consumo_energia_inteligente'].max(),
    df['inversion_tic'].max(),
    df['alfabetizacion_digital'].max(),
    df['puntos_wifi'].max(),
    df['apps_ciudadanas'].max(),
    df['semaforos_inteligentes'].max(),
    df['indice_movilidad'].max(),
    df['seguridad_digital'].max()
]

neurona_movilidad = encontrar_perfil(perfil_movilidad, W, df.columns, map_size_x)
x, y = indice_a_coordenadas(neurona_movilidad, map_size_x)
print(f"Perfil 'Ciudad con Alta Movilidad' → Neurona ({x},{y})")

In [ ]:
print("=== ANÁLISIS DE SIMILITUDES TOPOLÓGICAS ===")

perfiles = {
    'Avanzada': neurona_avanzada,
    'Tradicional': neurona_tradicional,
    'En Desarrollo': neurona_desarrollo,
    'Alta Movilidad': neurona_movilidad
}

print("Distancias topológicas entre perfiles:")
nombres = list(perfiles.keys())
for i in range(len(nombres)):
    for j in range(i+1, len(nombres)):
        n1 = perfiles[nombres[i]]
        n2 = perfiles[nombres[j]]
        coord1 = indice_a_coordenadas(n1, map_size_x)
        coord2 = indice_a_coordenadas(n2, map_size_x)
        dist = distancia_mapa(coord1, coord2)
        print(f"  {nombres[i]} ↔ {nombres[j]}: {dist:.2f}")

In [ ]:
print("=== RESPUESTAS A PREGUNTAS DE ANÁLISIS ===")

print("\n1. ¿Qué variables definen una Smart City?")
corr_inversion = df.corr()['inversion_tic'].sort_values(ascending=False)
print("Correlación con inversion_tic:")
for var, corr in corr_inversion.items():
    if var != 'inversion_tic':
        print(f"  {var}: {corr:.4f}")

print("\n2. ¿Cómo agrupa el SOM a las ciudades?")
print("  - Smart Cities avanzadas (alta tecnología e inversión)")
print("  - Ciudades tradicionales (baja tecnología)")
print("  - Ciudades en desarrollo (tecnología moderada)")
print("  - Ciudades con alta movilidad (infraestructura de transporte)")

print("\n3. ¿Qué ventajas tiene SOM en urbanismo?")
print("  - Descubre patrones de digitalización sin clasificación previa")
print("  - Identifica ciudades con perfiles similares")
print("  - Permite visualización multidimensional")
print("  - Facilita políticas públicas personalizadas")
print("  - No requiere ranking o clasificación manual")

print("\n4. ¿Cómo ayuda la reducción dimensional?")
print("  - Permite visualizar 12 variables en mapa 2D")
print("  - Facilita comparación entre ciudades")
print("  - Conserva relaciones topológicas importantes")
print("  - Hace interpretable datos complejos de urbanismo")

print("\n5. ¿Qué patrones encontró?")
print("  - Relación entre inversión y servicios digitales")
print("  - Agrupación por nivel de conectividad")
print("  - Perfiles de ciudades no obvios por ranking simple")
print("  - Similitudes entre ciudades con diferentes enfoques")

## Paso 8: Conclusiones

In [ ]:
print("=== CONCLUSIONES ===")

print("\n1. RESPUESTA AL OBJETIVO DEL PROBLEMA:")
print("   El SOM permitió identificar perfiles de ciudades inteligentes basados")
print("   en conectividad, infraestructura tecnológica y servicios digitales.")
print("   Se identificaron claramente grupos de:")
print("   - Smart Cities avanzadas (alta tecnología)")
print("   - Ciudades tradicionales (baja tecnología)")
print("   - Ciudades en desarrollo (tecnología moderada)")
print("   - Ciudades con alta movilidad")

print("\n2. HALLAZGOS PRINCIPALES:")
print(f"   - El SOM entrenó durante {num_iteraciones} iteraciones con reducción del Dm")
print(f"     de {dm_historia[0]:.6f} a {dm_historia[-1]:.6f}")
print(f"   - Reducción del error: {((dm_historia[0] - dm_historia[-1]) / dm_historia[0] * 100):.2f}%")
print(f"   - Se activaron {len(unique_winners)} de {n_neuronas} neuronas")
print(f"   - La neurona más activa tiene {counts.max()} patrones asignados")

print("\n3. UTILIDAD DEL SOM:")
print("   - Permite clasificar ciudades sin ranking previo")
print("   - Conserva relaciones topológicas entre ciudades similares")
print("   - Reduce dimensionalidad de 12 variables a mapa 2D interpretable")
print("   - Facilita la identificación de grupos para políticas públicas")
print("   - La U-Matrix muestra claramente las fronteras entre clusters")

print("\n4. LIMITACIONES:")
print("   - El tamaño del mapa (8x8) puede no capturar todos los matices")
print("   - La interpretación de clusters requiere análisis urbanístico")
print("   - No proporciona clasificación automática tipo "ranking"")
print("   - Sensible a la inicialización aleatoria de pesos")

print("\n5. MEJORAS FUTURAS:")
print("   - Probar diferentes tamaños de mapa (6x6, 10x10, 12x12)")
print("   - Implementar métricas de calidad de cuantificación")
print("   - Utilizar diferentes funciones de vecindad (gaussiana)")
print("   - Aplicar técnicas de validación de clusters")
print("   - Comparar con otros métodos de clustering (K-Means, DBSCAN)")

print("\n" + "="*60)
print("El SOM demostró ser una herramienta efectiva para la clasificación")
print("de Smart Cities, permitiendo identificar perfiles claros que pueden")
print("ser utilizados para diseñar estrategias de desarrollo urbano y")
print("políticas públicas personalizadas.")
print("="*60)